In [0]:
from pyspark.sql import SparkSession, DataFrame, Window
from datetime import datetime
from pyspark.sql.functions import to_timestamp, col, date_trunc, current_timestamp, to_date, concat_ws, sha2, row_number, trim, upper, when
from delta.tables import DeltaTable
import logging

In [0]:
def date_now() -> datetime:
    return datetime.strptime(datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "%Y-%m-%d %H:%M:%S")

In [0]:
dbutils.widgets.text("target_table_name", "", "Silver name")
dbutils.widgets.text("source_table_name", "", "Bronze name")
# dbutils.widgets.text("column_partition", "", "Column Partition")
dbutils.widgets.dropdown("environment", "dev", ["dev", "prod"])

silver_name = dbutils.widgets.get("target_table_name")
bronze_name = dbutils.widgets.get("source_table_name")
# column_partition = dbutils.widgets.get("column_partition")
environment = dbutils.widgets.get("environment")

In [0]:
source_path = f'`comercial-{environment}`.`bronze`.`{bronze_name}`'
target_path = f'`comercial-{environment}`.`silver`.`{silver_name}`'
logs_path = f'`comercial-{environment}`.`corporate`.`logs`'

In [0]:
spark = SparkSession.builder \
    .appName("Ingestion from csv to delta") \
    .getOrCreate()

logging.basicConfig(
    level=logging.INFO
    , format="%(asctime)s | %(levelname)s | %(message)s"
    , datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger(__name__)

In [0]:
df_bronze = spark.table(source_path)
display(df_bronze.select(
        *[
            when(col(column) == "", None)
                .otherwise(col(column))
            if dict(df_bronze.dtypes)[column] == "string" else col(column)
            for column in df_bronze.columns
        ]
    ))
# display(dict(df_bronze.dtypes))
# display(df_bronze.dtypes)
# display(df_bronze)

In [0]:
def apply_latest_update(df_bronze: DataFrame) -> DataFrame:
    df_bronze_partition = Window.partitionBy("hash_key").orderBy(col("updated_at").desc())
    df_bronze_with_row_number = df_bronze.withColumn("row_number", row_number().over(df_bronze_partition))
    return df_bronze_with_row_number.filter(col("row_number") == 1).drop("row_number")

def apply_string_treatment(df_bronze_latest: DataFrame) -> DataFrame:
    return df_bronze.select(
        *[
            trim(upper(column)) \
                .alias(column) 
            if dict(df_bronze.dtypes)[column] == "string" else col(column) 
            for column in df_bronze.columns
        ]
    )

def apply_null_treatment(latest_update: DataFrame) -> DataFrame:
    return latest_update.select(
        *[
            when(col(column) == "", None)
                .otherwise(col(column))
            if dict(latest_update.dtypes)[column] == "string" else col(column)
            for column in latest_update.columns
        ]
    )

def read_bronze() -> DataFrame: 
    df_bronze = spark.table(source_path)
    return df_bronze


In [0]:
def table_exists_in_catalog(table_name: str) -> bool:
    return spark.catalog.tableExists(table_name)

def get_delta_table(df_table_name: str) -> DeltaTable:
    return DeltaTable.forName(spark, df_table_name)

In [0]:
logs = {
    "dat_Hr_Process_Start": date_now(),
    "dat_Hr_Process_End":  '1900-01-01 00:00:00',
    "layer": "SILVER",
    "catalog": f'comercial-{environment}'.upper(),
    "source_Name": bronze_name.upper(),
    "target_Name": silver_name.upper(),
    "status": None,
    "count_Rows": None,
}

In [0]:
def merge_bronze_silver_table(bronze: DataFrame):
    if table_exists_in_catalog(target_path):
        get_delta_table(target_path) \
            .alias("target") \
            .merge(
                bronze.alias("source"),
                "source.hash_key = target.hash_key"
            ) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()
    else: 
        bronze.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_path)

def save_logs(logs):
    logs.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(logs_path) 

In [0]:

try:
    logger.info("===========Start Process===========")
    logger.info("Start reading bronze table")
    df_bronze = read_bronze()
    logger.info("End reading bronze table")

    logger.info("Start apply standard data quality")
    latest_update = apply_latest_update(df_bronze)
    df_bronze_with_null = apply_null_treatment(latest_update)
    df_bronze_standard = apply_string_treatment(df_bronze_with_null)
    logger.info("End apply standard data quality")

    logger.info("Start Ingestion Bronze")
    merge_bronze_silver_table(df_bronze_standard)
    logger.info("End Ingestion Bronze")
    
    logger.info("Start save logs")

    logs["status"] = "SUCCESS"
    logs["count_Rows"] = df_bronze.count()
    logs["dat_Hr_Process_End"] = date_now()

    df_logs = spark.createDataFrame([logs])
    
    save_logs(df_logs)
    logger.info("End saving logs")
    logger.info("===========End Process===========")
except Exception as e:
    logger.error(f'{e}')